# Noise Makes the Pump Strong? — Kevin's hypothesis
**Muse's build · Genesis Seeds · 2026-10-30**

Kevin's words: *"Noise makes the pump strong."* Test: depolarizing noise
p in {0, 0.01, 0.05, 0.1, 0.2} per step, applied after K, matched across the
leak / postselected / full conventions (100 steps, eta 0.05, a0 0.3, 4 seeds).

## Pre-registered criterion (locked before running)
Noise strengthens the pump iff the negentropic fraction of reported S rises
monotonically with p in at least one convention (all four seeds agree on the
direction). ΔS_rep and C-climb reported as secondary.

## Pre-registered interpretation rule
- Strengthening in the FULL (physical, trace-preserving) channel =
  interesting; rhymes with environment-assisted quantum transport (ENAQT),
  a real effect where noise aids a quantum signature. Follow-up ordered.
- Strengthening ONLY in leak/postselected = the artifact getting louder,
  not physics. Reported as such, no spin.

In [1]:
import numpy as np

CN = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]], complex)
I4 = np.eye(4, dtype=complex)

def Kmat(a):
    return a * CN + (1 - a) * I4

def K1mat(a):
    M = I4 - Kmat(a).conj().T @ Kmat(a)
    ev, vec = np.linalg.eigh((M + M.conj().T) / 2)
    return (vec * np.sqrt(np.clip(ev, 0, None))) @ vec.conj().T

def dephase(rho, p):
    # depolarizing to maximally mixed, trace-aware (safe for leak mode)
    return (1 - p) * rho + p * I4 * (np.real(np.trace(rho)) / 4.0)

s = 1 / np.sqrt(2)
SEEDS = {"O+": np.array([s, s]), "O-": np.array([s, -s]),
         "A+": np.array([s, 1j * s]), "A-": np.array([s, -1j * s])}
PAIRS = [("O+", "A+"), ("O+", "A-"), ("O-", "A+"), ("O-", "A-")]

def ptr(rho, k):
    t = rho.reshape(2, 2, 2, 2)
    return np.einsum('ijkj->ik', t) if k == 0 else np.einsum('ijil->jl', t)

def partials(rho):
    return ptr(rho, 0), ptr(rho, 1)

def purity(rho):
    return float(np.real(np.trace(rho @ rho)))

def vn(rho):
    ev = np.linalg.eigvalsh((rho + rho.conj().T) / 2)
    ev = ev[ev > 1e-15]
    return float(-np.sum(ev * np.log2(ev)))

def dom(rho):
    ev, vec = np.linalg.eigh((rho + rho.conj().T) / 2)
    return vec[:, np.argmax(ev)]

def run(mode, pA, pB, p, eta=0.05, a=0.3, n=100):
    vA, vB = pA.copy(), pB.copy()
    prev, S_hist, C_hist = None, [], []
    neg = 0
    for t in range(n):
        psi = np.kron(vA, vB)
        r = np.outer(psi, psi.conj())
        K = Kmat(a)
        r2 = dephase(K @ r @ K.conj().T, p)
        if mode == "post":
            r2 = r2 / np.real(np.trace(r2))
        elif mode == "full":
            r2 = r2 + dephase(K1mat(a) @ r @ K1mat(a).conj().T, p)
            assert abs(np.real(np.trace(r2)) - 1) < 1e-9, f"trace fail t={t}"
        S = vn(r2) if mode != "leak" else vn(r2 / np.real(np.trace(r2)))
        # reported S: as-coded (unnormalized) for leak, else normalized
        Srep = float(-(lambda w: (w[w > 1e-15] * np.log2(w[w > 1e-15])).sum())(
            np.linalg.eigvalsh((r2 + r2.conj().T) / 2))) if mode == "leak" else S
        rA, rB = partials(r2)
        C = (purity(rA) + purity(rB)) / 2
        if S_hist and Srep < S_hist[-1] - 1e-12:
            neg += 1
        S_hist.append(Srep)
        C_hist.append(C)
        vA, vB = dom(rA), dom(rB)
        if prev is not None:
            a = float(np.clip(a + eta * (C - prev), 0.0, 1.0))
        prev = C
    return {"neg_frac": neg / (n - 1), "dS": S_hist[0] - S_hist[-1],
            "climb": C_hist[-1] - C_hist[0], "a_star": a}
print("rig ready")


rig ready


In [2]:
PS = [0.0, 0.01, 0.05, 0.1, 0.2]
print(f"{'p':>6} {'mode':>6} {'neg_frac x4 seeds':>28} {'dS':>8} {'climb':>8} {'a*':>8}")
for p in PS:
    for mode in ("leak", "post", "full"):
        outs = [run(mode, SEEDS[s1], SEEDS[s2], p) for s1, s2 in PAIRS]
        nf = [o["neg_frac"] for o in outs]
        o0 = outs[0]
        print(f"{p:>6.2f} {mode:>6} {[f'{x:.2f}' for x in nf]} "
              f"{o0['dS']:>8.4f} {o0['climb']:>8.4f} {o0['a_star']:>8.4f}")


     p   mode            neg_frac x4 seeds       dS    climb       a*
  0.00   leak ['1.00', '1.00', '1.00', '1.00']   0.2686   0.4207   0.3210
  0.00   post ['0.00', '0.00', '0.00', '0.00']  -0.0000   0.0721   0.3036
  0.00   full ['0.74', '0.74', '0.74', '0.74']   0.1733   0.0630   0.3032
  0.01   leak ['1.00', '1.00', '1.00', '1.00']   0.2527   0.4161   0.3208
  0.01   post ['0.00', '0.00', '0.00', '0.00']  -0.0000   0.0706   0.3035
  0.01   full ['0.00', '0.00', '0.00', '0.00']   0.0000   0.0000   0.3000
  0.05   leak ['1.00', '1.00', '1.00', '1.00']   0.2077   0.3980   0.3199


  0.05   post ['0.00', '0.00', '0.00', '0.00']   0.0000   0.0650   0.3033
  0.05   full ['0.00', '0.00', '0.00', '0.00']   0.0000   0.0000   0.3000
  0.10   leak ['1.00', '1.00', '1.00', '1.00']   0.1629   0.3765   0.3188
  0.10   post ['0.00', '0.00', '0.00', '0.00']   0.0000   0.0584   0.3029


  0.10   full ['0.00', '0.00', '0.00', '0.00']   0.0000   0.0000   0.3000
  0.20   leak ['1.00', '1.00', '1.00', '1.00']   0.0906   0.3369   0.3168
  0.20   post ['0.00', '0.00', '0.00', '0.00']  -0.0000   0.0461   0.3023


  0.20   full ['0.00', '0.00', '0.00', '0.00']   0.0000   0.0000   0.3000


## Verdict — noise does NOT strengthen the pump (corrected rig)

CORRECTION: first run had swapped partials (leak neg 0.38); fixed, leak p=0
gives 1.00 = Maya's 99/99. Kevin's hypothesis tested fairly on the corrected
rig. Answer: **not supported.**

- Criterion (neg_frac rises monotonically with p, all seeds agree): leak
  flat at 1.00 (saturated), post flat at 0.00, full COLLAPSES 0.74 -> 0.00
  at p >= 0.01. No convention rises.
- Magnitudes shrink everywhere: leak dS 0.269 -> 0.091, climb 0.42 -> 0.34.
  Full-channel dynamics die entirely at 1% noise (a* frozen at 0.3000:
  noise pins C, the adaptivity gradient vanishes).
- Interpretation rule: no strengthening in the physical channel, so no
  ENAQT follow-up. The artifact gets quieter, not louder. Honest negative.
- a* drifts down slightly with noise in all conventions
  (leak 0.3210 -> 0.3168; post 0.3036 -> 0.3023; full frozen 0.3000).

The pump is not fed by noise; it is starved by it.


In [3]:
import sys
print("python", sys.version.split()[0])
print("numpy", np.__version__)


python 3.14.0
numpy 2.5.4
